# Open-weight model test for the still workflow

Runs the **same inputs** (prompt, negative, cropped look-sheet refs) that the approved `nano-banana-pro` recipe used, on open models,
so we can compare two-character consistency and hands/feet. Fictional adult characters only.

How to run: **Runtime > Change runtime type > pick the biggest GPU you have (A100 / L4)**, then **Runtime > Run all**. Nothing else to enter.
Costs Colab compute units only, no paid API. Expect ~15-25 min on first run (model download).

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout)

In [ ]:
%%bash
set -e
cd /content
[ -d new ] || git clone --depth 1 --branch cursor/ai-realistic-comic-e63b https://github.com/hailinsu-create/new.git
pip -q install pydantic-settings python-dotenv typer fal-client pyyaml 2>&1 | tail -2
pip -q install -U "git+https://github.com/huggingface/diffusers" transformers accelerate bitsandbytes sentencepiece 2>&1 | tail -2

In [ ]:
import json, os, sys
os.chdir("/content/new/ai-realistic-comic")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
from open_model_job import build_job
from pathlib import Path
STILLS = ["hades-persephone-throne", "baisuzhen-xuxian-coil"]
JOBS = {s: build_job(s, 0.55, Path(f"/content/work/{s}")) for s in STILLS}
for s, j in JOBS.items(): print(s, j["refs"])

In [ ]:
import torch, gc
from PIL import Image
OUT = Path("/content/out"); OUT.mkdir(exist_ok=True)
W, H = 960, 1280
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM {vram:.0f} GB")

def free():
    gc.collect(); torch.cuda.empty_cache()

def run_job(pipe, name, job, seeds=(1, 2), **kw):
    imgs = [Image.open(p).convert("RGB") for p in job["refs"]]
    for seed in seeds:
        g = torch.Generator("cuda").manual_seed(seed)
        r = pipe(image=imgs, prompt=job["prompt"], negative_prompt=job["negative"], height=H, width=W, generator=g, **kw)
        path = OUT / f"{job['still_id']}__{name}__s{seed}.png"
        r.images[0].save(path); print("saved", path)

## Model 1: Qwen-Image-Edit-2511 (Apache 2.0)

In [ ]:
if vram < 40:
    print(f"Skipping Qwen-Image-Edit-2511: needs an A100-class GPU (this one has {vram:.0f} GB).")
else:
    from diffusers import QwenImageEditPlusPipeline
    kw = dict(torch_dtype=torch.bfloat16)
    if vram < 70:
        from diffusers.quantizers import PipelineQuantizationConfig
        kw["quantization_config"] = PipelineQuantizationConfig(
            quant_backend="bitsandbytes_4bit",
            quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16},
            components_to_quantize=["transformer", "text_encoder"])
    pipe = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2511", **kw)
    pipe.enable_model_cpu_offload()
    for j in JOBS.values():
        run_job(pipe, "qwen2511", j, true_cfg_scale=4.0, guidance_scale=1.0, num_inference_steps=40)
    del pipe; free()

## Model 2: FLUX.2 klein 4B (Apache 2.0) - skipped automatically if it cannot load

In [ ]:
try:
    from diffusers import Flux2KleinPipeline
    pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    for j in JOBS.values():
        imgs = [Image.open(p).convert("RGB") for p in j["refs"]]
        for seed in (1, 2):
            g = torch.Generator("cuda").manual_seed(seed)
            im = pipe(image=imgs, prompt=j["prompt"], height=H, width=W, num_inference_steps=4, guidance_scale=1.0, generator=g).images[0]
            p = OUT / f"{j['still_id']}__flux2klein4b__s{seed}.png"; im.save(p); print("saved", p)
    del pipe; free()
except Exception:
    import traceback, subprocess
    traceback.print_exc()
    print(subprocess.run("pip list 2>/dev/null | grep -i -E '^(diffusers|transformers|torch|torchvision|huggingface-hub|peft|accelerate|bitsandbytes|numpy) '", shell=True, capture_output=True, text=True).stdout)

## Side-by-side: approved nano-banana-pro vs open models

In [ ]:
import matplotlib.pyplot as plt, glob
for s, j in JOBS.items():
    files = [j["approved_image"]] + sorted(glob.glob(str(OUT / f"{s}__*.png")))
    fig, axs = plt.subplots(1, len(files), figsize=(5 * len(files), 7))
    for a, f in zip(axs if len(files) > 1 else [axs], files):
        a.imshow(Image.open(f)); a.set_title(Path(f).stem.replace(s + "__", "") if "__" in f else "APPROVED nano-banana-pro", fontsize=9); a.axis("off")
    plt.show()

In [ ]:
!cd /content && zip -qr open_model_test.zip out && ls -la open_model_test.zip
from google.colab import files
files.download("/content/open_model_test.zip")